# Notebook 4 — Run 2 LLM Transaction Enrichment

Batch categorisation of bank transactions using Claude 3.5 Sonnet v2 via AWS Bedrock.
Improvements over Run 1: structured JSON output, DT pre-filter, biller/MCC lookups, non-spend regex engine, 7-field output schema.

**Sections:**
1. Setup & imports
2. BedrockClient & ExperimentTracker
3. Context loading (taxonomy, rules, patterns)
4. Pre-processing pipeline (DT, biller, MCC, regex)
5. Prompt builder
6. Sample build (stratified)
7. Batch enrichment loop
8. Evaluation (3-level accuracy)
9. Flag analysis & PO report

---
## 1. Setup & Imports

In [ ]:
import boto3
import json
import re
import time
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Optional

# Import all config — nothing hardcoded below this line
from run2_prompt_config import (
    MODEL_ID, MAX_TOKENS, TEMPERATURE, BUDGET_USD, TARGET_ROWS,
    BASE_LAYER_ROWS, TOP_UP_ROWS, TOP_UP_PER_CAT, FOCUS_CATEGORIES,
    AGGREGATOR_DT_VALUE, MCC_SKIP, MCC_NON_SPEND, BILLER_CODE_ZEROS,
    PROVIDER_ANZ, PROVIDER_ING, PROVIDER_BANKWEST, PROVIDER_UP, PROVIDER_MACQUARIE,
    INVALID_KEY_BLOCKLIST, MERCHANT_OVERRIDES, KEYWORD_OVERRIDES,
    DISAMBIGUATION_RULES, PROVIDER_NOTES, CONFIDENCE_DEFINITIONS, FLAG_TYPES,
    FEW_SHOT_TRANSACTION_IDS, RULES_FILE, RULES_STRIP_FROM,
    ASSET_DIR, OUTPUT_DIR, DATA_DIR
)

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 80)
print('Setup complete')
print(f'Model: {MODEL_ID} | Budget: ${BUDGET_USD} | Target rows: {TARGET_ROWS}')

---
## 2. BedrockClient & ExperimentTracker

In [ ]:
PRICING = {
    'anthropic.claude-3-5-sonnet-20241022-v2:0': {'input': 3.00, 'output': 15.00},
    'anthropic.claude-3-haiku-20240307-v1:0':    {'input': 0.25, 'output': 1.25},
}


class BedrockClient:
    def __init__(self, model_id: str = MODEL_ID, region: str = 'ap-southeast-2'):
        self.model_id = model_id
        self.provider = 'anthropic' if 'anthropic' in model_id else 'amazon'
        self.region   = region

    def invoke(self, prompt: str, system: str = None,
               max_tokens: int = MAX_TOKENS, temperature: float = TEMPERATURE,
               max_retries: int = 4) -> dict:
        bedrock = boto3.client('bedrock-runtime', region_name=self.region)
        body    = self._build_body(prompt, system, max_tokens, temperature)
        for attempt in range(max_retries):
            try:
                resp   = bedrock.invoke_model(
                    modelId=self.model_id, contentType='application/json',
                    accept='application/json', body=json.dumps(body)
                )
                result = self._parse_response(json.loads(resp['body'].read()))
                result['cost'] = self._calculate_cost(result['input_tokens'], result['output_tokens'])
                return result
            except Exception as e:
                if 'ThrottlingException' in str(type(e)) and attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    raise

    def _build_body(self, prompt, system, max_tokens, temperature):
        body = {
            'anthropic_version': 'bedrock-2023-05-31',
            'max_tokens': max_tokens, 'temperature': temperature,
            'messages': [{'role': 'user', 'content': prompt}]
        }
        if system:
            body['system'] = system
        return body

    def _parse_response(self, data):
        content = data['content'][0]['text']
        usage   = data.get('usage', {})
        inp     = usage.get('input_tokens', 0)
        out     = usage.get('output_tokens', 0)
        return {'content': content, 'input_tokens': inp,
                'output_tokens': out, 'total_tokens': inp + out}

    def _calculate_cost(self, inp, out):
        p = PRICING.get(self.model_id, {'input': 3.0, 'output': 15.0})
        return (inp / 1_000_000) * p['input'] + (out / 1_000_000) * p['output']


class ExperimentTracker:
    def __init__(self, name: str, budget_usd: float = BUDGET_USD):
        self.name       = name
        self.budget_usd = budget_usd
        self.experiments = []
        self._warned    = False

    def log(self, response: dict, notes: str = ''):
        total = self.total_cost() + response.get('cost', 0)
        if total >= self.budget_usd:
            raise RuntimeError(f'Budget ${self.budget_usd} exceeded — stopping.')
        if not self._warned and total >= self.budget_usd * 0.8:
            print(f'WARNING: 80% of budget used (${total:.4f} / ${self.budget_usd})')
            self._warned = True
        self.experiments.append({
            'timestamp':     datetime.now().isoformat(),
            'input_tokens':  response.get('input_tokens', 0),
            'output_tokens': response.get('output_tokens', 0),
            'cost_usd':      response.get('cost', 0),
            'notes':         notes,
        })

    def total_cost(self):   return sum(e['cost_usd'] for e in self.experiments)
    def total_tokens(self): return sum(e['input_tokens'] + e['output_tokens'] for e in self.experiments)
    def budget_status(self):
        s = self.total_cost()
        print(f'Budget: ${s:.4f} / ${self.budget_usd:.2f} ({s/self.budget_usd*100:.1f}%)')


client  = BedrockClient()
tracker = ExperimentTracker('run2a')
print(f'Client ready: {client.model_id}')

---
## 3. Context Loading

Load all static context once at startup.

In [ ]:
# ── Taxonomy ──────────────────────────────────────────────────────────────────
with open(OUTPUT_DIR / 'taxonomy_master.json') as f:
    taxonomy_json = json.load(f)

taxonomy_master = pd.read_csv(OUTPUT_DIR / 'taxonomy_master.csv')
VALID_KEYS      = set(taxonomy_master['category_key'].str.strip().str.upper())
TAXONOMY_CONTEXT = json.dumps(taxonomy_json, indent=2)

# ── Spend/non-spend decision rules (Sections 1+2 only) ────────────────────────
with open(ASSET_DIR / RULES_FILE) as f:
    raw_rules = f.read()
DECISION_RULES = raw_rules.split(RULES_STRIP_FROM)[0].strip()

# ── Bank patterns table ────────────────────────────────────────────────────────
bank_pt = pd.read_csv(ASSET_DIR / 'new_bank_patterns.csv')
bank_pt.columns = bank_pt.columns.str.strip().str.lower().str.replace(' ', '_')
BANK_PATTERNS_TABLE = bank_pt.to_string(index=False)

# ── Lookup maps ───────────────────────────────────────────────────────────────
biller_df  = pd.read_csv(ASSET_DIR / 'biller_mapping.csv')
mcc_df     = pd.read_csv(ASSET_DIR / 'mcc_mapping.csv')
ns_rules   = pd.read_csv(ASSET_DIR / 'layered_ns_rules.csv')

biller_map = dict(zip(
    biller_df['biller_code'].astype(str).str.strip(),
    biller_df['base_category_key'].str.strip().str.upper()
))
mcc_map = dict(zip(
    mcc_df['mcc_code'].astype(str).str.strip(),
    mcc_df['base_category_key'].str.strip().str.upper()
))

print(f'Taxonomy: {len(VALID_KEYS)} valid keys')
print(f'Rules:    {len(DECISION_RULES):,} chars ({len(DECISION_RULES)//4:,} est. tokens)')
print(f'Taxonomy context: {len(TAXONOMY_CONTEXT):,} chars ({len(TAXONOMY_CONTEXT)//4:,} est. tokens)')
print(f'Biller map: {len(biller_map)} entries | MCC map: {len(mcc_map)} entries')
print(f'NS regex rules: {len(ns_rules)} rows')

---
## 4. Pre-Processing Pipeline

In [ ]:
def _str(val) -> str:
    """Safe string coerce — treats None/nan/empty as empty string."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ''
    return str(val).strip()


def run_decision_tree(row: dict) -> tuple[str, str]:
    """
    Python Decision Tree — mirrors production BC logic.
    Only applies when aggregator == AGGREGATOR_DT_VALUE.
    Returns (hint, rule_label) where hint in {spend, non_spend, none}.
    """
    agg = _str(row.get('aggregator_id', row.get('aggregator', '')))
    if agg != AGGREGATOR_DT_VALUE:
        return 'none', None

    tt      = _str(row.get('transaction_type', ''))
    mcc     = _str(row.get('merchant_category_code', ''))
    bc      = _str(row.get('biller_code', ''))
    bn      = _str(row.get('biller_name', ''))
    mn      = _str(row.get('merchant_name', ''))
    prov    = _str(row.get('provider_name', '')).upper()
    amount  = row.get('amount', 0)
    try:
        amount = float(amount)
    except (ValueError, TypeError):
        amount = 0.0

    bc_zero = bc in BILLER_CODE_ZEROS
    mcc_ok  = mcc and mcc not in MCC_SKIP

    # Rules applied top-down, first match wins
    if tt in {'0', '1', '2'}:                              return 'non_spend', 'Rule 1'
    if bc and not bc_zero:                                 return 'spend',     'Rule 2'
    if bn:                                                 return 'spend',     'Rule 3'
    if mcc in MCC_NON_SPEND:                               return 'non_spend', 'Rule 4'
    if mcc and amount == 0.0:                              return 'non_spend', 'Rule 5'
    if mcc_ok and PROVIDER_MACQUARIE not in prov:          return 'spend',     'Rule 6'
    if PROVIDER_ANZ in prov and mn:                        return 'spend',     'Rule 9'
    if PROVIDER_ANZ in prov and bc_zero:                   return 'non_spend', 'Rule 10'
    if PROVIDER_ING in prov and mn:                        return 'spend',     'Rule 11'
    if PROVIDER_BANKWEST in prov and mn:                   return 'spend',     'Rule 12'
    if PROVIDER_UP in prov and mn:                         return 'spend',     'Rule 13'
    return 'none', None


def resolve_biller(row: dict, bmap: dict) -> Optional[str]:
    """Look up biller_code in biller_mapping. Returns category key or None."""
    bc = _str(row.get('biller_code', ''))
    if bc in BILLER_CODE_ZEROS:
        return None
    return bmap.get(bc)


def resolve_mcc(row: dict, mmap: dict) -> Optional[str]:
    """Look up MCC in mcc_mapping. Excludes non-definitive codes."""
    mcc = _str(row.get('merchant_category_code', ''))
    if mcc in MCC_SKIP:
        return None
    return mmap.get(mcc)


def run_ns_regex(row: dict, rules: pd.DataFrame) -> Optional[str]:
    """
    Non-spend regex engine (M1func logic).
    Only meaningful when dt_hint == non_spend.
    Returns category_type string or None.
    """
    desc = _str(row.get('original_description', '')).upper()
    for _, rule in rules.iterrows():
        pattern = _str(rule.get('pattern', ''))
        cat     = _str(rule.get('base_category_key', '')).upper()
        if not pattern or not cat:
            continue
        try:
            if re.search(pattern, desc, re.IGNORECASE):
                return cat
        except re.error:
            continue
    return None


def enrich_signals(row: dict) -> dict:
    """Run full pre-processing pipeline. Returns signals dict for prompt injection."""
    dt_hint, dt_rule    = run_decision_tree(row)
    biller_cat          = resolve_biller(row, biller_map)
    mcc_cat             = resolve_mcc(row, mcc_map)
    regex_cat_type      = run_ns_regex(row, ns_rules) if dt_hint == 'non_spend' else None
    return {
        'dt_hint':            dt_hint,
        'dt_rule':            dt_rule or 'none',
        'biller_category':    biller_cat or 'none',
        'mcc_category':       mcc_cat or 'none',
        'regex_category_type': regex_cat_type or 'none',
    }


print('Pre-processing functions ready')

---
## 5. Prompt Builder

System prompt is built once. User prompt is built per transaction.

In [ ]:
## INSTRUCTION
## In Notebook 4, Section 5 "Prompt Builder", replace the ENTIRE first cell
## (the one that defines build_system_prompt) with this code.
## The only change vs the original is:
##   - The closing .format(...) call is REMOVED
##   - The flag line at the bottom is simplified (no {flag_str} placeholder)
##   - Everything else is identical
## Root cause: .format() was choking on {17,18,19,20,21,24} in DISAMBIGUATION_RULES

def build_system_prompt() -> str:
    blocklist_str   = ', '.join(INVALID_KEY_BLOCKLIST)
    merchant_str    = '\n'.join(f'  {k} -> {v}' for k, v in MERCHANT_OVERRIDES.items())
    keyword_str     = '\n'.join(f'  {k} -> {v}' for k, v in KEYWORD_OVERRIDES.items())
    disambig_str    = '\n'.join(f'  {k}: {v}' for k, v in DISAMBIGUATION_RULES.items())
    provider_str    = '\n'.join(f'  {k}: {v}' for k, v in PROVIDER_NOTES.items())
    confidence_str  = '\n'.join(f'  {k}: {v}' for k, v in CONFIDENCE_DEFINITIONS.items())
    flag_str        = '\n'.join(f'  - {f}' for f in FLAG_TYPES)

    # NOTE: pure f-string — NO .format() call at the end.
    # .format() was failing because DISAMBIGUATION_RULES contains {17,18,19,20,21,24}
    # which Python's format engine misinterprets as a placeholder.
    return f"""You are a financial transaction categorisation engine for an Australian bank data platform.
Your task: assign a single base_category_key to each transaction from the approved taxonomy below.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HOW TO CATEGORISE — FOLLOW THIS ORDER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Use your own understanding of merchants, brands, and language as your primary tool.
Pre-filter signals in the user prompt are structured hints — they narrow your search space
but do not replace your reasoning.

Step 1 — Read and apply pre-filter signals in priority order:
  1. biller_category   <- deterministic BPAY lookup (highest trust)
  2. mcc_category      <- MCC code lookup (high trust)
  3. dt_hint           <- Decision Tree output (high trust)
  4. regex_category_type <- non-spend pattern engine (medium trust, non-spend only)

  If signals agree -> follow them, set confidence = high.
  If signals conflict -> follow highest-priority signal, set flag type = rule_conflict.
  If all signals are none -> reason independently from transaction fields.

Step 2 — Navigate the taxonomy top-down:
  (a) Determine spend_type: spend or non_spend
  (b) Spend -> identify category_group, then select base_category_key within that group
  (c) Non-spend -> identify category_type, then select base_category_key within that type

Step 3 — Apply language understanding for final key selection.
  Amount sign: negative = debit (money out), positive = credit (money in).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SPEND / NON-SPEND DECISION RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{DECISION_RULES}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PROVIDER-SPECIFIC NOTES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{provider_str}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BANK PATTERN OVERRIDES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{BANK_PATTERNS_TABLE}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
GUARDRAILS — KNOWN FAILURE CASES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Merchant mappings (apply before general reasoning):
{merchant_str}

Keyword mappings (apply to original_description):
{keyword_str}

Disambiguation rules:
{disambig_str}

NEVER return these — they are not valid base_category_key values:
  {blocklist_str}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONFIDENCE DEFINITIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{confidence_str}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FLAG TYPES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{flag_str}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
APPROVED TAXONOMY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{TAXONOMY_CONTEXT}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FEW-SHOT EXAMPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
--- Example 1: Woolworths -> GROCERIES ---
Input:  provider=CBA | description=WOOLWORTHS 1234 SYDNEY | merchant_name=Woolworths | amount=-62.40 | transaction_type=7
Signals: dt_hint=spend (Rule 6) | mcc_category=GROCERIES | biller_category=none
Output: {{"spend_type":"spend","category_group":"Food & Drink","category_type":null,"base_category_key":"GROCERIES","category_key_method":"direct","confidence":"high","reason":"Woolworths is a known supermarket; MCC confirms GROCERIES.","flag":null}}

--- Example 2: Chemist Warehouse -> HEALTHCARE_MEDICAL ---
Input:  provider=CBA | description=CHEMIST WAREHOUSE 0123 SYDNEY | merchant_name=Chemist Warehouse | amount=-34.99 | transaction_type=7
Signals: dt_hint=spend (Rule 9) | mcc_category=HEALTHCARE_MEDICAL | biller_category=none
Output: {{"spend_type":"spend","category_group":"Health & Beauty","category_type":null,"base_category_key":"HEALTHCARE_MEDICAL","category_key_method":"direct","confidence":"high","reason":"Chemist Warehouse is a known pharmacy; MCC confirms HEALTHCARE_MEDICAL.","flag":null}}

--- Example 3: Bunnings -> HOME_RENOVATION___MAINTENANCE ---
Input:  provider=Westpac | description=BUNNINGS WAREHOUSE AUBURN | merchant_name=Bunnings | amount=-87.40 | transaction_type=5
Signals: dt_hint=spend (Rule 6) | mcc_category=HOME_RENOVATION___MAINTENANCE | biller_category=none
Output: {{"spend_type":"spend","category_group":"Home","category_type":null,"base_category_key":"HOME_RENOVATION___MAINTENANCE","category_key_method":"direct","confidence":"high","reason":"Bunnings is a known hardware/home improvement retailer.","flag":null}}

--- Example 4: Hotel venue -> BAR not RESTAURANT ---
Input:  provider=Up | description=THE ROSE AND CROWN HOTEL MELB | merchant_name=Rose & Crown Hotel | amount=-42.00 | transaction_type=7
Signals: dt_hint=spend (Rule 13) | mcc_category=none | biller_category=none
Output: {{"spend_type":"spend","category_group":"Food & Drink","category_type":null,"base_category_key":"BAR","category_key_method":"direct","confidence":"medium","reason":"Venue name contains Hotel indicating a licensed venue, not a restaurant.","flag":null}}

--- Example 5: Macquarie internal transfer -> INTERNAL_TRANSFER ---
Input:  provider=Macquarie | description=To linked account 001234567 | merchant_name=To linked account 001234567 | amount=-500.00 | transaction_type=3
Signals: dt_hint=none (aggregator!=7) | mcc_category=none | biller_category=none | regex_category_type=none
Output: {{"spend_type":"non_spend","category_group":null,"category_type":"TRANSFERS","base_category_key":"INTERNAL_TRANSFER","category_key_method":"derived","confidence":"medium","reason":"Macquarie pattern: To linked account -> INTERNAL_TRANSFER.","flag":null}}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Respond with ONLY a valid JSON object. No preamble, no markdown fences.
Fields:
  spend_type           : "spend" | "non_spend"
  category_group       : string (spend only) | null
  category_type        : string (non-spend only) | null
  base_category_key    : string — MUST be a valid key from the taxonomy above
  category_key_method  : "derived" | "direct"
  confidence           : "high" | "medium" | "low"
  reason               : one sentence
  flag                 : null | {{"type": "<one of the flag types listed above>", "detail": "one sentence"}}
"""


def build_user_prompt(row: dict, signals: dict) -> str:
    return (
        f"Transaction:\n"
        f"  provider:                {_str(row.get('provider_name'))}\n"
        f"  description:             {_str(row.get('original_description'))}\n"
        f"  merchant_name:           {_str(row.get('merchant_name'))}\n"
        f"  amount:                  {row.get('amount')}\n"
        f"  transaction_type:        {_str(row.get('transaction_type'))}\n"
        f"  account_type:            {_str(row.get('account_type'))}\n"
        f"  merchant_category_code:  {_str(row.get('merchant_category_code'))}\n"
        f"  biller_code:             {_str(row.get('biller_code'))}\n"
        f"  biller_name:             {_str(row.get('biller_name'))}\n"
        f"\nPre-filter signals:\n"
        f"  biller_category:      {signals['biller_category']}\n"
        f"  mcc_category:         {signals['mcc_category']}\n"
        f"  dt_hint:              {signals['dt_hint']}\n"
        f"  dt_rule:              {signals['dt_rule']}\n"
        f"  regex_category_type:  {signals['regex_category_type']}\n"
    )


SYSTEM_PROMPT = build_system_prompt()
est_sys_tokens = len(SYSTEM_PROMPT) // 4
print(f'System prompt built: ~{est_sys_tokens:,} tokens')
print('Check this is reasonable before running batch (Run 1 was ~3,015 tokens)')

---
## 6. Sample Build

Stratified draw: base layer (provider coverage) + focus top-up.

In [ ]:
df_full = pd.read_parquet(DATA_DIR / 'df_sample_small.parquet')
print(f'Source: {len(df_full):,} rows | {df_full["provider_name"].nunique()} providers')

rng = np.random.default_rng(seed=42)

# ── Base layer: ~19 rows per provider ─────────────────────────────────────────
n_providers  = df_full['provider_name'].nunique()
per_provider = max(1, BASE_LAYER_ROWS // n_providers)

base_frames = []
for prov, grp in df_full.groupby('provider_name'):
    n = min(len(grp), per_provider)
    base_frames.append(grp.sample(n, random_state=42))
df_base = pd.concat(base_frames).drop_duplicates(subset='transaction_id')

# ── Top-up layer: focus categories ────────────────────────────────────────────
topup_frames = []
for cat in FOCUS_CATEGORIES:
    pool = df_full[
        (df_full['ground_truth'] == cat) &
        (~df_full['transaction_id'].isin(df_base['transaction_id']))
    ]
    n = min(len(pool), TOP_UP_PER_CAT)
    if n > 0:
        topup_frames.append(pool.sample(n, random_state=42))
        print(f'  {cat}: {n} rows')
    else:
        print(f'  {cat}: 0 rows (none available outside base layer)')

df_topup  = pd.concat(topup_frames).drop_duplicates(subset='transaction_id') if topup_frames else pd.DataFrame()

# ── Combine, dedup, exclude few-shot IDs ──────────────────────────────────────
df_sample = (
    pd.concat([df_base, df_topup])
    .drop_duplicates(subset='transaction_id')
    .loc[lambda d: ~d['transaction_id'].isin(FEW_SHOT_TRANSACTION_IDS)]
    .sample(frac=1, random_state=42)  # shuffle
    .reset_index(drop=True)
)

print(f'\nFinal sample: {len(df_sample):,} rows')
print(f'  Base: {len(df_base)} | Top-up: {len(df_topup)} | After dedup/exclusion: {len(df_sample)}')
print(df_sample['provider_name'].value_counts().to_string())

---
## 7. Batch Enrichment Loop

Results saved incrementally after every row. Safe to re-run from last checkpoint.

In [ ]:
RESULTS_PATH = OUTPUT_DIR / 'run2a_results.csv'

# ── Resume support: skip already-processed rows ───────────────────────────────
if RESULTS_PATH.exists():
    df_done      = pd.read_csv(RESULTS_PATH)
    done_ids     = set(df_done['transaction_id'].astype(str))
    df_remaining = df_sample[~df_sample['transaction_id'].astype(str).isin(done_ids)]
    print(f'Resuming: {len(done_ids)} done, {len(df_remaining)} remaining')
else:
    df_done      = pd.DataFrame()
    df_remaining = df_sample.copy()
    print(f'Starting fresh: {len(df_remaining)} rows to process')

print(f'Estimated cost: ~${len(df_remaining) * 0.0012:.2f} (based on Run 1 avg $0.0012/row)')

In [ ]:
print(client.model_id)

In [ ]:
def parse_llm_response(raw: str) -> dict:
    """Parse JSON from LLM response. Returns dict with error key on failure."""
    # Strip markdown fences if present
    clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip(), flags=re.MULTILINE)
    try:
        return json.loads(clean)
    except json.JSONDecodeError as e:
        return {'parse_error': str(e), 'raw': raw[:200]}


errors, skipped = [], []
write_header = not RESULTS_PATH.exists()

for i, (_, row) in enumerate(df_remaining.iterrows()):
    try:
        tracker.budget_status() if i % 50 == 0 and i > 0 else None

        signals      = enrich_signals(row.to_dict())
        user_prompt  = build_user_prompt(row.to_dict(), signals)
        response     = client.invoke(user_prompt, system=SYSTEM_PROMPT)
        parsed       = parse_llm_response(response['content'])

        bck          = str(parsed.get('base_category_key', '')).strip().upper()
        valid        = bck in VALID_KEYS
        gt           = str(row.get('ground_truth', '')).strip().upper()
        match        = bck == gt

        record = {
            # ── Identity
            'transaction_id':      row.get('transaction_id'),
            'provider_name':       row.get('provider_name'),
            'transaction_type':    row.get('transaction_type'),
            'account_type':        row.get('account_type'),
            'amount':              row.get('amount'),
            'original_description': row.get('original_description'),
            'merchant_name':       row.get('merchant_name'),
            'ground_truth':        gt,
            # ── Pre-filter signals
            'dt_hint':             signals['dt_hint'],
            'dt_rule':             signals['dt_rule'],
            'biller_category':     signals['biller_category'],
            'mcc_category':        signals['mcc_category'],
            'regex_category_type': signals['regex_category_type'],
            # ── LLM output
            'llm_spend_type':      parsed.get('spend_type'),
            'llm_category_group':  parsed.get('category_group'),
            'llm_category_type':   parsed.get('category_type'),
            'base_category_key':   bck,
            'category_key_method': parsed.get('category_key_method'),
            'confidence':          parsed.get('confidence'),
            'reason':              parsed.get('reason'),
            'flag_type':           parsed.get('flag', {}).get('type') if parsed.get('flag') else None,
            'flag_detail':         parsed.get('flag', {}).get('detail') if parsed.get('flag') else None,
            'parse_error':         parsed.get('parse_error'),
            # ── Eval
            'valid_key':           valid,
            'match':               match,
            # ── Cost
            'input_tokens':        response['input_tokens'],
            'output_tokens':       response['output_tokens'],
            'cost':                response['cost'],
        }

        # Incremental save
        pd.DataFrame([record]).to_csv(
            RESULTS_PATH, mode='a', header=write_header, index=False
        )
        write_header = False
        tracker.log(response, notes=f'row {i}')

        if (i + 1) % 25 == 0:
            print(f'[{i+1}/{len(df_remaining)}] cost so far: ${tracker.total_cost():.4f}')

    except RuntimeError as e:
        # Budget exceeded
        print(f'STOPPED: {e}')
        break
    except Exception as e:
        print(f'ERROR row {i} ({row.get("transaction_id")}): {e}')
        errors.append({'transaction_id': row.get('transaction_id'), 'error': str(e)})
        continue

print(f'\nDone. Total cost: ${tracker.total_cost():.4f} | Errors: {len(errors)}')
tracker.budget_status()

---
## 8. Evaluation — 3-Level Accuracy

In [ ]:
df = pd.read_csv(RESULTS_PATH)
print(f'Loaded {len(df):,} results')

# ── Taxonomy lookups for Level 2 ──────────────────────────────────────────────
gt_meta = (
    taxonomy_master
    .assign(k=lambda x: x['category_key'].str.strip().str.upper())
    .set_index('k')[['spend_type', 'category_group', 'category_type']]
)

def lookup_group_type(key):
    row = gt_meta.loc[key] if key in gt_meta.index else None
    if row is None:
        return None, None, None
    return row['spend_type'], row.get('category_group'), row.get('category_type')

df[['gt_spend_type', 'gt_group', 'gt_cat_type']] = [
    lookup_group_type(k) for k in df['ground_truth']
]

# ── Level 1: spend_type ───────────────────────────────────────────────────────
df['l1_match'] = df['llm_spend_type'] == df['gt_spend_type']

# ── Level 2: category_group / category_type ───────────────────────────────────
def l2_match(row):
    if row['gt_spend_type'] == 'spend':
        return row['llm_category_group'] == row['gt_group']
    else:
        return row['llm_category_type'] == row['gt_cat_type']

df['l2_match'] = df.apply(l2_match, axis=1)

# ── Level 3: exact key ────────────────────────────────────────────────────────
# match column already set in batch loop

# ── Headlines ─────────────────────────────────────────────────────────────────
n = len(df)
print('=' * 50)
print('RUN 2a — ACCURACY RESULTS')
print('=' * 50)
print(f'Total rows:        {n:,}')
print(f'Parse errors:      {df["parse_error"].notna().sum():,}')
print(f'Invalid keys:      {(~df["valid_key"]).sum():,} ({(~df["valid_key"]).mean()*100:.1f}%)')
print(f'')
print(f'L1 spend_type:     {df["l1_match"].mean()*100:.1f}%')
print(f'L2 group/type:     {df["l2_match"].mean()*100:.1f}%')
print(f'L3 exact key ★:    {df["match"].mean()*100:.1f}%  (target: >60%)')
print(f'')
print(f'Total cost:        ${df["cost"].sum():.4f}')
print(f'Cost per row:      ${df["cost"].mean():.6f}')

In [ ]:
# ── Accuracy by confidence tier ───────────────────────────────────────────────
print('\nAccuracy by confidence tier:')
print(
    df.groupby('confidence')['match']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'accuracy', 'count': 'n'})
    .assign(accuracy=lambda x: (x['accuracy'] * 100).round(1))
    .to_string()
)

In [ ]:
# ── Accuracy by provider ──────────────────────────────────────────────────────
print('\nAccuracy by provider:')
print(
    df.groupby('provider_name')['match']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'accuracy', 'count': 'n'})
    .assign(accuracy=lambda x: (x['accuracy'] * 100).round(1))
    .sort_values('accuracy')
    .to_string()
)

In [ ]:
# ── Accuracy by category ──────────────────────────────────────────────────────
print('\nAccuracy by ground truth category (worst first):')
print(
    df.groupby('ground_truth')['match']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'accuracy', 'count': 'n'})
    .assign(accuracy=lambda x: (x['accuracy'] * 100).round(1))
    .sort_values(['accuracy', 'n'])
    .to_string()
)

In [ ]:
# ── DT hint agreement ─────────────────────────────────────────────────────────
print('\nDT hint agreement with LLM spend_type:')
dt_rows = df[df['dt_hint'].isin(['spend', 'non_spend'])].copy()
dt_rows['dt_agree'] = dt_rows['dt_hint'] == dt_rows['llm_spend_type']
print(f'  Rows with DT hint: {len(dt_rows)}')
print(f'  Agreement rate:    {dt_rows["dt_agree"].mean()*100:.1f}%')
print('\nDisagreements by dt_rule:')
print(
    dt_rows[~dt_rows['dt_agree']]
    .groupby('dt_rule').size()
    .sort_values(ascending=False)
    .to_string()
)

---
## 9. Flag Analysis & PO Report

In [ ]:
flagged = df[df['flag_type'].notna()].copy()
print(f'Flagged transactions: {len(flagged)} ({len(flagged)/len(df)*100:.1f}%)')
print('\nFlag type distribution:')
print(flagged['flag_type'].value_counts().to_string())

In [ ]:
# ── Taxonomy gap report ───────────────────────────────────────────────────────
gaps = flagged[flagged['flag_type'] == 'taxonomy_gap'][
    ['transaction_id', 'provider_name', 'original_description',
     'ground_truth', 'base_category_key', 'flag_detail']
].copy()

print(f'\nTaxonomy gaps: {len(gaps)}')
display(gaps.head(20))

In [ ]:
# ── Ambiguous boundary report ─────────────────────────────────────────────────
ambig = flagged[flagged['flag_type'] == 'ambiguous_boundary'][
    ['transaction_id', 'provider_name', 'original_description',
     'ground_truth', 'base_category_key', 'flag_detail']
].copy()

print(f'\nAmbiguous boundaries: {len(ambig)}')
display(ambig.head(20))

In [ ]:
# ── Export PO report ──────────────────────────────────────────────────────────
PO_REPORT_PATH = OUTPUT_DIR / 'run2a_po_flag_report.csv'
flagged.to_csv(PO_REPORT_PATH, index=False)
print(f'PO flag report saved: {PO_REPORT_PATH} ({len(flagged)} rows)')

# Full results
FULL_RESULTS_PATH = OUTPUT_DIR / 'run2a_results_final.csv'
df.to_csv(FULL_RESULTS_PATH, index=False)
print(f'Full results saved: {FULL_RESULTS_PATH} ({len(df)} rows)')